# <a id='toc1_'></a>[Capturing Alternate Abbreviation Symbols Experiment](#toc0_)

**Table of contents**<a id='toc0_'></a>    
- [Capturing Alternate Abbreviation Symbols Experiment](#toc1_)    
    - [Import manually curated table](#toc1_1_1_)    
    - [Add the official gene name for each sample from HGNC](#toc1_1_2_)    
    - [Add the abstract and full text for each sample](#toc1_1_3_)    
      - [Not all contexts are programmatically accessible](#toc1_1_3_1_)    
    - [Using details from each sample, generate a custom prompt for each sample](#toc1_1_4_)    
      - [Prompt with just the abstract as context](#toc1_1_4_1_)    
      - [Prompt with full text as context](#toc1_1_4_2_)    
    - [Run prompt through Claude](#toc1_1_5_)    
      - [Run a test with 2 samples (abstracts as context)](#toc1_1_5_1_)    
      - [Run all samples (abstracts as context)](#toc1_1_5_2_)    
      - [Run a test with 2 samples (full text as context)](#toc1_1_5_3_)    
      - [Run a test with 20 samples (full text as context)](#toc1_1_5_4_)    
      - [Whole set with full text as context](#toc1_1_5_5_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

Goal of this experiment is to see how well Claude can match my annotations of gene alias pairs in literary context

In [1]:
import ssl
import certifi
import os

os.environ["SSL_CERT_FILE"] = certifi.where()
ssl._create_default_https_context = ssl.create_default_context


In [2]:
import pandas as pd
import requests
from Bio import Entrez
from bs4 import BeautifulSoup
from tqdm import tqdm
tqdm.pandas()
import boto3
import json
import time
from functools import lru_cache
from urllib.error import HTTPError
from http.client import IncompleteRead
import numpy as np

### <a id='toc1_1_1_'></a>[Import manually curated table](#toc0_)

This is a file where I manually annotated gene alias pairs by answering the following questions:
- Does this alias symbol represent this gene in this publication?
- How does this alias symbol represent this gene in this publication?

In [3]:
#Alternate Abbreviation Symbol Capture- Manually Curated Set
alt_abbrev_annotation_df = pd.read_csv(
    "../input/Alternate Abbreviation Symbol Capture- Manually Curated Set.csv", skip_blank_lines=True)
alt_abbrev_annotation_df

,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,captured,captured as:,PMIDs from Pubtator3,Does alias represent gene in this publication?,How? I,How? II,"If not this gene, what is it?",Text Location,Relevant Text (surrounding),Notes
0,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9477305,yes,Other,mouse phenotype biomarker,-,discussion,Mice homozygous for the tub (rd5) mutation exh...,NaN
1,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9390831,yes,Other,mouse phenotype biomarker,-,abstract,Mice homozygous for a defect of the tub (rd5) ...,NaN
2,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,7479945,yes,Other,mouse phenotype biomarker,-,abstract,We report a mouse model of type I Usher syndro...,NaN
3,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,39961042,no,-,-,primer,methods,The primers used in this study were: RD5: 5′–A...,NaN
4,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,38980911,no,-,-,primer,methods,PCR amplification of DNA was performed using a...,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
122,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,9458019,no,-,-,blood antibodies,methods,Incompatibility in ABO blood group antigens wa...,NaN
123,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,18822025,no,-,-,α1-adrenergic receptor blocker,introduction,The AUA recommends α1-adrenergic receptor bloc...,NaN
124,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,26761027,no,-,-,genotype of rhinovirus,introduction,"A1A, A1B, A2, A23, A25, A29, A30, A31, A44, A4...",NaN
125,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,31837028,no,-,-,red blood cell type,methods,We used in vitro phagocytosis by monocytes and...,NaN


### <a id='toc1_1_2_'></a>[Add the official gene name for each sample from HGNC](#toc0_)

In [4]:
def get_gene_name(hgnc_id):
    """ Retreive official gene name from HGNC for each sample in the set.

    :param hgnc_id: The HGNC ID associated with the primary gene symbol of each sample
    :return: The official gene name from HGNC for the passed HGNC ID
    """
    url = f"https://rest.genenames.org/fetch/hgnc_id/{hgnc_id}"
    headers = {"Accept": "application/json"}  # Request JSON format
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        data = response.json()
        try:
            gene_name = data['response']['docs'][0]['name']
            return gene_name
        except IndexError:
            return f"No gene found for HGNC ID {hgnc_id}"
    else:
        return f"Error: {response.status_code}"

In [5]:
gene_cache = {}

def get_gene_name_cached(hgnc_id):
    """ Retrieve the official gene name from HGNC using a cache to avoid repeated API requests for the same HGNC ID.

    :param hgnc_id: The HGNC ID associated with the primary gene symbol
    :return: The official gene name from HGNC for the passed HGNC ID
    """
    if hgnc_id not in gene_cache:
        gene_cache[hgnc_id] = get_gene_name(hgnc_id)
    return gene_cache[hgnc_id]

In [6]:
alt_abbrev_annotation_df["gene_name"] = alt_abbrev_annotation_df["HGNC_ID"].apply(get_gene_name_cached)
alt_abbrev_annotation_df

,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,captured,captured as:,PMIDs from Pubtator3,Does alias represent gene in this publication?,How? I,How? II,"If not this gene, what is it?",Text Location,Relevant Text (surrounding),Notes,gene_name
0,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9477305,yes,Other,mouse phenotype biomarker,-,discussion,Mice homozygous for the tub (rd5) mutation exh...,NaN,TUB bipartite transcription factor
1,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9390831,yes,Other,mouse phenotype biomarker,-,abstract,Mice homozygous for a defect of the tub (rd5) ...,NaN,TUB bipartite transcription factor
2,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,7479945,yes,Other,mouse phenotype biomarker,-,abstract,We report a mouse model of type I Usher syndro...,NaN,TUB bipartite transcription factor
3,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,39961042,no,-,-,primer,methods,The primers used in this study were: RD5: 5′–A...,NaN,TUB bipartite transcription factor
4,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,38980911,no,-,-,primer,methods,PCR amplification of DNA was performed using a...,NaN,TUB bipartite transcription factor
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
122,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,9458019,no,-,-,blood antibodies,methods,Incompatibility in ABO blood group antigens wa...,NaN,alpha-1-B glycoprotein
123,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,18822025,no,-,-,α1-adrenergic receptor blocker,introduction,The AUA recommends α1-adrenergic receptor bloc...,NaN,alpha-1-B glycoprotein
124,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,26761027,no,-,-,genotype of rhinovirus,introduction,"A1A, A1B, A2, A23, A25, A29, A30, A31, A44, A4...",NaN,alpha-1-B glycoprotein
125,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,31837028,no,-,-,red blood cell type,methods,We used in vitro phagocytosis by monocytes and...,NaN,alpha-1-B glycoprotein


### <a id='toc1_1_3_'></a>[Add the abstract and full text for each sample](#toc0_)

In [7]:
Entrez.email = "anastasia.bratulin@nationwidechildrens.org"

# Throttling (≤3 requests/sec without API key)
NCBI_SLEEP = 0.34

# Retry policy
MAX_RETRIES = 5
BASE_BACKOFF = 1.0

In [8]:
def _entrez_call(func):
    """Execute an Entrez call with retry, exponential backoff, and throttling."""
    for attempt in range(MAX_RETRIES):
        try:
            time.sleep(NCBI_SLEEP)
            return func()
        except (RuntimeError, HTTPError, IncompleteRead) as e:
            if attempt == MAX_RETRIES - 1:
                return None
            time.sleep(BASE_BACKOFF * (2 ** attempt))


In [9]:
@lru_cache(maxsize=10000)
def get_abstract(pmid):
    """ Retrieve the abstract text for a PubMed article given its PMID. The abstract is fetched from the NCBI PubMed database using Entrez.
    
    If the abstract consists of multiple sections, they are concatenated
    into a single string separated by blank lines.

    :param pmid: The PubMed ID (PMID) of the article
    :return: The abstract text as a string, or None if no abstract is available
    """
    pmid = str(pmid)

    def call():
        handle = Entrez.efetch(
            db="pubmed", id=pmid, rettype="abstract", retmode="xml"
        )
        records = Entrez.read(handle)
        handle.close()
        return records

    records = _entrez_call(call)
    if not records:
        return None

    try:
        abstract = (
            records["PubmedArticle"][0]
            ["MedlineCitation"]["Article"]["Abstract"]["AbstractText"]
        )
        return "\n\n".join(abstract) if isinstance(abstract, list) else abstract
    except (KeyError, IndexError, TypeError):
        return None


In [10]:
@lru_cache(maxsize=10000)
def get_full_text_paragraphs(pmid):
    """ Retrieve full-text paragraphs for a PubMed article when available via PMC.

    This function attempts to map a PubMed ID (PMID) to a PubMed Central ID
    (PMCID). If a PMCID exists and the article allows XML access, the full text
    is fetched from PMC in XML format and parsed into individual paragraph
    strings.

    :param pmid: pmid : The PubMed ID (PMID) of the article.
    :return: A list of paragraph texts extracted from the PMC full text, or None
    if the article is not available in PMC, access is restricted, or an
    error occurs during retrieval.
    """
    pmid = str(pmid)

    # Step 1: PMID → PMCID
    def elink_call():
        handle = Entrez.elink(dbfrom="pubmed", db="pmc", id=pmid)
        records = Entrez.read(handle)
        handle.close()
        return records

    records = _entrez_call(elink_call)
    if not records:
        return None

    try:
        pmcid = records[0]["LinkSetDb"][0]["Link"][0]["Id"]
    except (IndexError, KeyError, TypeError):
        return None

    # Step 2: Fetch PMC XML
    def efetch_call():
        handle = Entrez.efetch(
            db="pmc", id=pmcid, rettype="full", retmode="xml"
        )
        xml = handle.read()
        handle.close()
        return xml

    xml = _entrez_call(efetch_call)
    if not xml:
        return None

    if b"does not allow downloading of the full text in XML form" in xml:
        return None

    try:
        soup = BeautifulSoup(xml, "xml")
        paragraphs = [p.get_text(strip=True) for p in soup.find_all("p")]
        return paragraphs or None
    except Exception:
        return None


In [11]:
def chunk_paragraphs(paragraphs, max_chars=20000):
    """ Chunk paragraph text into size-limited chunks.
    Paragraph boundaries are preserved; paragraphs are not split
    across chunks.

    :param paragraphs: A list of paragraph strings to be grouped into chunks
    :param max_chars: Maximum number of characters allowed per chunk
    :return: A list of text chunks, each containing one or more paragraphs
    """
    chunks = []
    current = ""

    for p in paragraphs:
        if len(current) + len(p) <= max_chars:
            current += p + "\n\n"
        else:
            chunks.append(current.strip())
            current = p + "\n\n"

    if current:
        chunks.append(current.strip())

    return chunks


In [12]:
def get_full_text_chunks(pmid, max_chars=20000):
    """ Retrieve chunked full-text content of a PubMed article via PMC.

    This function retrieves the full-text paragraphs for a given PubMed ID
    (PMID) when available via PubMed Central (PMC), and groups those paragraphs
    into size-limited text chunks suitable for downstream processing such as
    LLM prompting.

    :param pmid: The PubMed ID (PMID) of the article
    :param max_chars: Maximum number of characters allowed per text chunk
    :return: A list of text chunks derived from the article full text, or None
             if the article is unavailable, restricted, or an error occurs
    """
    try:
        paragraphs = get_full_text_paragraphs(pmid)
        if not paragraphs:
            return None
        return chunk_paragraphs(paragraphs, max_chars=max_chars)
    except Exception:
        return None


In [13]:
alt_abbrev_annotation_using_fulltext_df = alt_abbrev_annotation_df.copy()


alt_abbrev_annotation_using_fulltext_df["pmid_full_text_chunks"] = (
    alt_abbrev_annotation_using_fulltext_df["PMIDs from Pubtator3"]
    .progress_apply(get_full_text_chunks)
)

100%|██████████| 127/127 [03:55<00:00,  1.86s/it]


In [14]:
alt_abbrev_annotation_using_fulltext_df = (
    alt_abbrev_annotation_using_fulltext_df
    .explode("pmid_full_text_chunks")
    .rename(columns={"pmid_full_text_chunks": "pmid_full_text_chunk"})
)
alt_abbrev_annotation_using_fulltext_df    

,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,captured,captured as:,PMIDs from Pubtator3,Does alias represent gene in this publication?,How? I,How? II,"If not this gene, what is it?",Text Location,Relevant Text (surrounding),Notes,gene_name,pmid_full_text_chunk
0,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9477305,yes,Other,mouse phenotype biomarker,-,discussion,Mice homozygous for the tub (rd5) mutation exh...,NaN,TUB bipartite transcription factor,None
1,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9390831,yes,Other,mouse phenotype biomarker,-,abstract,Mice homozygous for a defect of the tub (rd5) ...,NaN,TUB bipartite transcription factor,These authors contributed equally to this work...
1,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9390831,yes,Other,mouse phenotype biomarker,-,abstract,Mice homozygous for a defect of the tub (rd5) ...,NaN,TUB bipartite transcription factor,"In this paper, we confirm that biallelic inact..."
2,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,7479945,yes,Other,mouse phenotype biomarker,-,abstract,We report a mouse model of type I Usher syndro...,NaN,TUB bipartite transcription factor,None
3,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,39961042,no,-,-,primer,methods,The primers used in this study were: RD5: 5′–A...,NaN,TUB bipartite transcription factor,Blastocystissp. is a zoonotic intestinal proto...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,26761027,no,-,-,genotype of rhinovirus,introduction,"A1A, A1B, A2, A23, A25, A29, A30, A31, A44, A4...",NaN,alpha-1-B glycoprotein,Rhinoviruses (RVs) and respiratory enterovirus...
124,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,26761027,no,-,-,genotype of rhinovirus,introduction,"A1A, A1B, A2, A23, A25, A29, A30, A31, A44, A4...",NaN,alpha-1-B glycoprotein,Phylogeny of the different human enterovirus s...
124,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,26761027,no,-,-,genotype of rhinovirus,introduction,"A1A, A1B, A2, A23, A25, A29, A30, A31, A44, A4...",NaN,alpha-1-B glycoprotein,RVs are known to optimally grow at cooler temp...
125,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,31837028,no,-,-,red blood cell type,methods,We used in vitro phagocytosis by monocytes and...,NaN,alpha-1-B glycoprotein,Immunoglobulin therapy including intravenous i...


In [75]:
print(alt_abbrev_annotation_using_fulltext_df['pmid_full_text_chunk'].iloc[1:3])

1    These authors contributed equally to this work...
1    In this paper, we confirm that biallelic inact...
Name: pmid_full_text_chunk, dtype: object


In [16]:
alt_abbrev_annotation_using_abstract_df = alt_abbrev_annotation_df.copy()

alt_abbrev_annotation_using_abstract_df["abstract_pmid_text"] = (
    alt_abbrev_annotation_using_abstract_df["PMIDs from Pubtator3"]
    .progress_apply(get_abstract)
)
alt_abbrev_annotation_using_abstract_df

100%|██████████| 127/127 [01:13<00:00,  1.74it/s]


,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,captured,captured as:,PMIDs from Pubtator3,Does alias represent gene in this publication?,How? I,How? II,"If not this gene, what is it?",Text Location,Relevant Text (surrounding),Notes,gene_name,abstract_pmid_text
0,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9477305,yes,Other,mouse phenotype biomarker,-,discussion,Mice homozygous for the tub (rd5) mutation exh...,NaN,TUB bipartite transcription factor,Some genetic syndromes causing loss of hearing...
1,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9390831,yes,Other,mouse phenotype biomarker,-,abstract,Mice homozygous for a defect of the tub (rd5) ...,NaN,TUB bipartite transcription factor,Mice homozygous for a defect of the tub (rd5) ...
2,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,7479945,yes,Other,mouse phenotype biomarker,-,abstract,We report a mouse model of type I Usher syndro...,NaN,TUB bipartite transcription factor,Usher syndrome is a group of diseases with aut...
3,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,39961042,no,-,-,primer,methods,The primers used in this study were: RD5: 5′–A...,NaN,TUB bipartite transcription factor,Blastocystis sp. is a zoonotic intestinal prot...
4,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,38980911,no,-,-,primer,methods,PCR amplification of DNA was performed using a...,NaN,TUB bipartite transcription factor,Blastocystis is a unicellular eukaryote common...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
122,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,9458019,no,-,-,blood antibodies,methods,Incompatibility in ABO blood group antigens wa...,NaN,alpha-1-B glycoprotein,Despite great efforts to promote the donation ...
123,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,18822025,no,-,-,α1-adrenergic receptor blocker,introduction,The AUA recommends α1-adrenergic receptor bloc...,NaN,alpha-1-B glycoprotein,To evaluate the safety profile and efficacy of...
124,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,26761027,no,-,-,genotype of rhinovirus,introduction,"A1A, A1B, A2, A23, A25, A29, A30, A31, A44, A4...",NaN,alpha-1-B glycoprotein,Rhinoviruses (RVs) and respiratory enterovirus...
125,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,31837028,no,-,-,red blood cell type,methods,We used in vitro phagocytosis by monocytes and...,NaN,alpha-1-B glycoprotein,Immunoglobulin therapy including intravenous i...


#### <a id='toc1_1_3_1_'></a>[Not all contexts are programmatically accessible](#toc0_)

In [17]:
num_missing_abstracts = len(
    alt_abbrev_annotation_using_abstract_df.loc[
        alt_abbrev_annotation_using_abstract_df["abstract_pmid_text"].isna()
    ]
)

num_missing_full_texts = len(
    alt_abbrev_annotation_using_fulltext_df.loc[
        alt_abbrev_annotation_using_fulltext_df["pmid_full_text_chunk"].isna()
    ]
)

print(f"Number of abstracts that are not accessible: {num_missing_abstracts}")
print(f"Number of full texts that are not accessible: {num_missing_full_texts}")

Number of abstracts that are not accessible: 3
Number of full texts that are not accessible: 14


### <a id='toc1_1_4_'></a>[Using details from each sample, generate a custom prompt for each sample](#toc0_)

In [18]:
def generate_alt_abbrev_prompt(alias_symbol, hgnc_id, primary_gene_symbol, name):
    """ Generate an LLM prompt for curating alias gene symbols and determining their relationship to an official HGNC gene record.

    The generated prompt instructs the model to:
    - Validate whether an alias symbol corresponds to the provided HGNC record ID
    - Classify the relationship as an Alias, Alternate Abbreviation, or Other
    - Extract exact textual evidence from a biomedical context
    - Return the result as a strictly formatted JSON object

    :param alias_symbol: The alias gene symbol to be evaluated
    :param hgnc_id: The HGNC record ID associated with the primary gene
    :param primary_gene_symbol: The official HGNC gene symbol
    :param name: The official HGNC gene name
    :return: A formatted prompt string for use with a language model
    """
    prompt = f"""Role:You are a biomedical gene nomenclature curator trained in HGNC gene identity and alias resolution.

    Inputs:
    Alias gene symbol: {alias_symbol}
    HGNC record ID: {hgnc_id}
    Primary gene symbol: {primary_gene_symbol}
    Official HGNC gene name: {name}
    Biomedical text context (may include PMID)

    Task:
    Determine whether the alias symbol in the given text refers to the gene represented by the provided HGNC record ID.

    Rules:
    If the text does not support that the alias refers to the provided gene, return:
    "alias_symbol_represents_primary_gene": "NO"
    "relationship_type": ""
    "evidence": ""

    If the text does support the gene identity, set "alias_symbol_represents_primary_gene": "YES" and assign exactly one relationship:
    ALIAS → official gene symbol explicitly appears in the text with the alias
    ALIAS — POSITIVE EXAMPLE
    Text: “TUB (MIM #601197), also known as rd5 or RDOB, maps to 11p15.4 and encodes the tubby-bipartite transcription regulator”
    Alias gene symbol: rd5
    Primary gene symbol: TUB

    ALIAS — NEGATIVE EXAMPLE
    Text: “The RBD-Gly proteins with one RBD comprise RBM3 and hnRNP G in human and a series of exceptionally Gly-rich proteins in plant.”
    Alias gene symbol: HNRNP-G
    Primary gene symbol: RBMXP1

    ALTERNATE ABBREVIATION → alias is expanded to a phrase identically matching the official HGNC gene name
    ALTERNATE ABBREVIATION — POSITIVE EXAMPLE
    Text: “Alpha 1-beta glycoprotein (A1B) was purified to homogeneity from human plasma using a three-step procedure involving pseudo-ligand affinity chromatography on immobilized Cibacron Blue 3-GA, gel filtration chromatography on Sephadex G-200 and ion-exchange chromatography on DEAE Affigel Blue.”
    Alias gene symbol: A1B
    Primary gene symbol: A1BG

    ALTERNATE ABBREVIATION — NEGATIVE EXAMPLE
    Text: “Based on the N‐terminal domain, the NLRs are classified into four sub‐families: the baculoviral inhibitory repeat‐like domain (NLRB), the acidic transactivation domain (NLRA), the caspase activation and recruitment domain (CARD; NLRC) and the pyrin domain (NLRP).”
    Alias gene symbol: NLRA
    Primary gene symbol: CIITA

    OTHER → gene identity is clear but neither rule above applies
    OTHER — POSITIVE EXAMPLE
    Text: “Primary monoclonal antibodies anti-ABP1 [EPR24299-52] (ab278497) and anti-tubulin [YL172] (ab 6160) were obtained from ABCAM (Cambridge, UK), while HRP-linked anti-rat and anti-rabbit IgG were obtained from Thermo Fisher Scientific (Rockford, IL, USA) and from Cell Signaling Technology (Waltham, MA, USA), respectively.”
    Alias gene symbol: ABP1
    Primary gene symbol: AOC1

    OTHER — NEGATIVE EXAMPLE
    Text: “Therefore, a dual-branch global extraction module is proposed to extract detailed local information and rich global information. which includes a global attention branch (GAB) and a global compensation branch (GCB).”
    Alias gene symbol: GAB
    Primary gene symbol: A1BG

    Evidence must be one exact sentence copied verbatim from the text.
    Do not infer relationships or paraphrase evidence.


    Output (Strict):
    Output only one JSON object. No explanations. No markdown.
    {{
    "pmid": "",
    "alias_symbol": "",
    "primary_gene_symbol": "",
    "name": "",
    "alias_symbol_represents_primary_gene": "YES or NO",
    "relationship_type": "ALIAS or ALTERNATE ABBREVIATION or OTHER or empty string",
    "evidence": ""
    }}"""


    return prompt

#### <a id='toc1_1_4_1_'></a>[Prompt with just the abstract as context](#toc0_)

In [19]:
alt_abbrev_annotation_with_prompt_using_abstract_df = alt_abbrev_annotation_using_abstract_df.copy()
alt_abbrev_annotation_with_prompt_using_abstract_df['context'] = None
alt_abbrev_annotation_with_prompt_using_abstract_df['alt_abbrev_prompt'] = None

In [20]:
for idx, row in alt_abbrev_annotation_with_prompt_using_abstract_df.iterrows():
    alias_symbol = row['gene_symbol']
    primary_gene_symbol = row['primary_gene_symbol']
    name = row['gene_name']
    hgnc_id = row['HGNC_ID']
    prompt_base = generate_alt_abbrev_prompt(alias_symbol, hgnc_id, primary_gene_symbol, name)

    context = f'PMID: {row['PMIDs from Pubtator3']}\n Abstract: {row['abstract_pmid_text']}'

    full_prompt = f'{prompt_base}\n\n{context}'

    alt_abbrev_annotation_with_prompt_using_abstract_df.at[idx,'context'] = context
    alt_abbrev_annotation_with_prompt_using_abstract_df.at[idx,'alt_abbrev_prompt'] = full_prompt    



alt_abbrev_annotation_with_prompt_using_abstract_df.head()

,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,captured,captured as:,PMIDs from Pubtator3,Does alias represent gene in this publication?,How? I,How? II,"If not this gene, what is it?",Text Location,Relevant Text (surrounding),Notes,gene_name,abstract_pmid_text,context,alt_abbrev_prompt
0,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9477305,yes,Other,mouse phenotype biomarker,-,discussion,Mice homozygous for the tub (rd5) mutation exh...,NaN,TUB bipartite transcription factor,Some genetic syndromes causing loss of hearing...,PMID: 9477305\n Abstract: Some genetic syndrom...,Role:You are a biomedical gene nomenclature cu...
1,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9390831,yes,Other,mouse phenotype biomarker,-,abstract,Mice homozygous for a defect of the tub (rd5) ...,NaN,TUB bipartite transcription factor,Mice homozygous for a defect of the tub (rd5) ...,PMID: 9390831\n Abstract: Mice homozygous for ...,Role:You are a biomedical gene nomenclature cu...
2,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,7479945,yes,Other,mouse phenotype biomarker,-,abstract,We report a mouse model of type I Usher syndro...,NaN,TUB bipartite transcription factor,Usher syndrome is a group of diseases with aut...,PMID: 7479945\n Abstract: Usher syndrome is a ...,Role:You are a biomedical gene nomenclature cu...
3,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,39961042,no,-,-,primer,methods,The primers used in this study were: RD5: 5′–A...,NaN,TUB bipartite transcription factor,Blastocystis sp. is a zoonotic intestinal prot...,PMID: 39961042\n Abstract: Blastocystis sp. is...,Role:You are a biomedical gene nomenclature cu...
4,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,38980911,no,-,-,primer,methods,PCR amplification of DNA was performed using a...,NaN,TUB bipartite transcription factor,Blastocystis is a unicellular eukaryote common...,PMID: 38980911\n Abstract: Blastocystis is a u...,Role:You are a biomedical gene nomenclature cu...


#### <a id='toc1_1_4_2_'></a>[Prompt with full text as context](#toc0_)

In [86]:
alt_abbrev_annotation_with_prompt_using_fulltext_df = alt_abbrev_annotation_using_fulltext_df.copy()
alt_abbrev_annotation_with_prompt_using_fulltext_df['context'] = None
alt_abbrev_annotation_with_prompt_using_fulltext_df['alt_abbrev_prompt'] = None

In [87]:
alt_abbrev_annotation_with_prompt_using_fulltext_df = (
    alt_abbrev_annotation_with_prompt_using_fulltext_df.reset_index(drop=True)
)

for idx, row in alt_abbrev_annotation_with_prompt_using_fulltext_df.iterrows():
    alias_symbol = row['gene_symbol']
    primary_gene_symbol = row['primary_gene_symbol']
    name = row['gene_name']
    hgnc_id = row['HGNC_ID']
    prompt_base = generate_alt_abbrev_prompt(alias_symbol, hgnc_id, primary_gene_symbol, name)

    context = f'PMID: {row['PMIDs from Pubtator3']}\n Full Text: {row['pmid_full_text_chunk']}'

    full_prompt = f'{prompt_base}\n\n{context}'

    alt_abbrev_annotation_with_prompt_using_fulltext_df.at[idx,'context'] = context
    alt_abbrev_annotation_with_prompt_using_fulltext_df.at[idx,'alt_abbrev_prompt'] = full_prompt    



alt_abbrev_annotation_with_prompt_using_fulltext_df.head()

,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,captured,captured as:,PMIDs from Pubtator3,Does alias represent gene in this publication?,How? I,How? II,"If not this gene, what is it?",Text Location,Relevant Text (surrounding),Notes,gene_name,pmid_full_text_chunk,context,alt_abbrev_prompt
0,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9477305,yes,Other,mouse phenotype biomarker,-,discussion,Mice homozygous for the tub (rd5) mutation exh...,NaN,TUB bipartite transcription factor,None,PMID: 9477305\n Full Text: None,Role:You are a biomedical gene nomenclature cu...
1,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9390831,yes,Other,mouse phenotype biomarker,-,abstract,Mice homozygous for a defect of the tub (rd5) ...,NaN,TUB bipartite transcription factor,These authors contributed equally to this work...,PMID: 9390831\n Full Text: These authors contr...,Role:You are a biomedical gene nomenclature cu...
2,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9390831,yes,Other,mouse phenotype biomarker,-,abstract,Mice homozygous for a defect of the tub (rd5) ...,NaN,TUB bipartite transcription factor,"In this paper, we confirm that biallelic inact...","PMID: 9390831\n Full Text: In this paper, we c...",Role:You are a biomedical gene nomenclature cu...
3,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,7479945,yes,Other,mouse phenotype biomarker,-,abstract,We report a mouse model of type I Usher syndro...,NaN,TUB bipartite transcription factor,None,PMID: 7479945\n Full Text: None,Role:You are a biomedical gene nomenclature cu...
4,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,39961042,no,-,-,primer,methods,The primers used in this study were: RD5: 5′–A...,NaN,TUB bipartite transcription factor,Blastocystissp. is a zoonotic intestinal proto...,PMID: 39961042\n Full Text: Blastocystissp. is...,Role:You are a biomedical gene nomenclature cu...


### <a id='toc1_1_5_'></a>[Run prompt through Claude](#toc0_)

login via aws sso login in terminal

In [24]:
# Initialize the Bedrock Runtime client
bedrock = boto3.client("bedrock-runtime", region_name="us-east-2")

# Replace with your actual inference profile ID or ARN
INFERENCE_PROFILE_ID = "us.anthropic.claude-3-5-sonnet-20240620-v1:0"


def query_claude_sonnet(prompt: str) -> str:
    """ Send a prompt to the Claude Sonnet model via Amazon Bedrock and return the generated response text.

    :param prompt: The input prompt to send to the Claude Sonnet model
    :return: The model-generated response text, or an error message string
    """

    body = {
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "max_tokens": 1024,
        "temperature": 0.0,
        "anthropic_version": "bedrock-2023-05-31"
    }

    try:
        response = bedrock.invoke_model(
            body=json.dumps(body),
            modelId=INFERENCE_PROFILE_ID,
            contentType="application/json",
            accept="application/json"
        )
        response_body = json.loads(response["body"].read())
        return response_body["content"][0]["text"]
    except Exception as e:
        return f"[Error] {str(e)}"

#### <a id='toc1_1_5_1_'></a>[Run a test with 2 samples (abstracts as context)](#toc0_)

In [25]:
test_alt_abbrev_annotation_with_prompt_using_abstract_df = alt_abbrev_annotation_with_prompt_using_abstract_df.iloc[:2]
test_alt_abbrev_annotation_with_prompt_using_abstract_df

,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,captured,captured as:,PMIDs from Pubtator3,Does alias represent gene in this publication?,How? I,How? II,"If not this gene, what is it?",Text Location,Relevant Text (surrounding),Notes,gene_name,abstract_pmid_text,context,alt_abbrev_prompt
0,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9477305,yes,Other,mouse phenotype biomarker,-,discussion,Mice homozygous for the tub (rd5) mutation exh...,NaN,TUB bipartite transcription factor,Some genetic syndromes causing loss of hearing...,PMID: 9477305\n Abstract: Some genetic syndrom...,Role:You are a biomedical gene nomenclature cu...
1,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9390831,yes,Other,mouse phenotype biomarker,-,abstract,Mice homozygous for a defect of the tub (rd5) ...,NaN,TUB bipartite transcription factor,Mice homozygous for a defect of the tub (rd5) ...,PMID: 9390831\n Abstract: Mice homozygous for ...,Role:You are a biomedical gene nomenclature cu...


about 8s ea

In [26]:
# #Comment out after run!
# test_alt_abbrev_annotation_with_prompt_using_abstract_df['alt_abbrev_response'] = None

# for idx, row in test_alt_abbrev_annotation_with_prompt_using_abstract_df.iterrows():
#     # Skip rows where there is no available abstract
#     if pd.isna(row['abstract_pmid_text']):
#         continue

#     test_alt_abbrev_annotation_with_prompt_using_abstract_df.at[
#         idx, 'alt_abbrev_response'
#     ] = query_claude_sonnet(row['alt_abbrev_prompt'])

#     print(f'{idx} Done')

In [27]:
# #Comment out after run!
# test_alt_abbrev_annotation_with_prompt_using_abstract_df.to_excel('../output/test_alt_abbrev_annotation_with_prompt_using_abstract_df.xlsx')

In [28]:
test_alt_abbrev_annotation_with_prompt_using_abstract_df = pd.read_excel("../output/test_alt_abbrev_annotation_with_prompt_using_abstract_df.xlsx")

In [29]:
print(test_alt_abbrev_annotation_with_prompt_using_abstract_df['alt_abbrev_response'][0])

{
    "pmid": "9477305",
    "alias_symbol": "rd5",
    "primary_gene_symbol": "TUB",
    "name": "TUB bipartite transcription factor",
    "alias_symbol_represents_primary_gene": "YES",
    "relationship_type": "ALIAS",
    "evidence": "Mice homozygous for the tub (rd5) mutation exhibit progressive retinal degeneration, sensorineural hearing loss, reduced fertility, and obesity, and presently represent the only animal model with neuroepithelial degeneration of both cochlea and retina without other neurological abnormalities."
}


In [30]:
print(test_alt_abbrev_annotation_with_prompt_using_abstract_df['alt_abbrev_response'][1])

{
    "pmid": "9390831",
    "alias_symbol": "rd5",
    "primary_gene_symbol": "TUB",
    "name": "TUB bipartite transcription factor",
    "alias_symbol_represents_primary_gene": "YES",
    "relationship_type": "ALIAS",
    "evidence": "Mice homozygous for a defect of the tub (rd5) gene exhibit cochlear and retinal degeneration combined with obesity, and resemble certain human autosomal recessive sensory deficit syndromes."
}


In [31]:
len(alt_abbrev_annotation_with_prompt_using_abstract_df)

127

#### <a id='toc1_1_5_2_'></a>[Run all samples (abstracts as context)](#toc0_)

In [32]:
# #Comment out after run!
# alt_abbrev_annotation_with_prompt_using_abstract_df['alt_abbrev_response'] = None

# for idx, row in alt_abbrev_annotation_with_prompt_using_abstract_df.iterrows():
#     # Skip rows where there is no available abstract
#     if pd.isna(row['abstract_pmid_text']):
#         continue

#     alt_abbrev_annotation_with_prompt_using_abstract_df.at[
#         idx, 'alt_abbrev_response'
#     ] = query_claude_sonnet(row['alt_abbrev_prompt'])

#     print(f'{idx} Done')

In [33]:
# #Comment out after run!
# alt_abbrev_annotation_with_prompt_using_abstract_df.to_excel('../output/alt_abbrev_annotation_with_prompt_using_abstract_df.xlsx')

In [34]:
alt_abbrev_annotation_with_prompt_using_abstract_df = pd.read_excel("../output/alt_abbrev_annotation_with_prompt_using_abstract_df.xlsx")

In [35]:
print(alt_abbrev_annotation_with_prompt_using_abstract_df['alt_abbrev_response'][126])

{
    "pmid": "2447112",
    "alias_symbol": "A1B",
    "primary_gene_symbol": "A1BG",
    "name": "alpha-1-B glycoprotein",
    "alias_symbol_represents_primary_gene": "YES",
    "relationship_type": "ALTERNATE ABBREVIATION",
    "evidence": "Alpha 1-beta glycoprotein (A1B) was purified to homogeneity from human plasma using a three-step procedure involving pseudo-ligand affinity chromatography on immobilized Cibacron Blue 3-GA, gel filtration chromatography on Sephadex G-200 and ion-exchange chromatography on DEAE Affigel Blue."
}


#### <a id='toc1_1_5_3_'></a>[Run a test with 2 samples (full text as context)](#toc0_)

In [36]:
test_alt_abbrev_annotation_with_prompt_using_fulltext_df = alt_abbrev_annotation_with_prompt_using_fulltext_df.iloc[:2]
test_alt_abbrev_annotation_with_prompt_using_fulltext_df

,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,captured,captured as:,PMIDs from Pubtator3,Does alias represent gene in this publication?,How? I,How? II,"If not this gene, what is it?",Text Location,Relevant Text (surrounding),Notes,gene_name,pmid_full_text_chunk,context,alt_abbrev_prompt
0,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9477305,yes,Other,mouse phenotype biomarker,-,discussion,Mice homozygous for the tub (rd5) mutation exh...,NaN,TUB bipartite transcription factor,None,PMID: 9477305\n Full Text: None,Role:You are a biomedical gene nomenclature cu...
1,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9390831,yes,Other,mouse phenotype biomarker,-,abstract,Mice homozygous for a defect of the tub (rd5) ...,NaN,TUB bipartite transcription factor,These authors contributed equally to this work...,"PMID: 9390831\n Full Text: In this paper, we c...",Role:You are a biomedical gene nomenclature cu...


In [37]:
# #Comment out after run!
# test_alt_abbrev_annotation_with_prompt_using_fulltext_df['alt_abbrev_response'] = None

# for idx, row in test_alt_abbrev_annotation_with_prompt_using_fulltext_df.iterrows():
#     # Skip rows where there is no available full text
#     if pd.isna(row['pmid_full_text_chunk']):
#         continue

#     test_alt_abbrev_annotation_with_prompt_using_fulltext_df.at[
#         idx, 'alt_abbrev_response'
#     ] = query_claude_sonnet(row['alt_abbrev_prompt'])

#     print(f'{idx} Done')

In [38]:
# #Comment out after run!
# test_alt_abbrev_annotation_with_prompt_using_fulltext_df.to_excel('../output/test_alt_abbrev_annotation_with_prompt_using_fulltext_df.xlsx',index=False)

In [39]:
test_alt_abbrev_annotation_with_prompt_using_fulltext_df = pd.read_excel("../output/test_alt_abbrev_annotation_with_prompt_using_fulltext_df.xlsx")

In [40]:
print(test_alt_abbrev_annotation_with_prompt_using_fulltext_df['alt_abbrev_response'][0])

nan


In [41]:
print(test_alt_abbrev_annotation_with_prompt_using_fulltext_df['alt_abbrev_response'][1])

{
    "pmid": "9390831",
    "alias_symbol": "rd5",
    "primary_gene_symbol": "TUB",
    "name": "TUB bipartite transcription factor",
    "alias_symbol_represents_primary_gene": "YES",
    "relationship_type": "OTHER",
    "evidence": "Characterization of the pathogenic TUB variant."
}


In [42]:
test_alt_abbrev_annotation_with_prompt_using_fulltext_df

,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,captured,captured as:,PMIDs from Pubtator3,Does alias represent gene in this publication?,How?,"If not this gene, what is it?",Text Location,Relevant Text (surrounding),Notes,gene_name,pmid_full_text_chunk,context,alt_abbrev_prompt,alt_abbrev_response
0,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9477305,yes,mouse phenotype biomarker,-,discussion,Mice homozygous for the tub (rd5) mutation exh...,NaN,TUB bipartite transcription factor,NaN,PMID: 9477305\n Full Text: None,You are an expert biomedical scientist and gen...,NaN
1,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9390831,yes,mouse phenotype biomarker,-,abstract,Mice homozygous for a defect of the tub (rd5) ...,NaN,TUB bipartite transcription factor,These authors contributed equally to this work...,"PMID: 9390831\n Full Text: Conceptualization, ...",You are an expert biomedical scientist and gen...,"{\n ""pmid"": ""9390831"",\n ""alias_symbol"":..."


#### <a id='toc1_1_5_4_'></a>[Run a test with 20 samples (full text as context)](#toc0_)

In [43]:
test20_alt_abbrev_annotation_with_prompt_using_fulltext_df = alt_abbrev_annotation_with_prompt_using_fulltext_df.iloc[:20]
test20_alt_abbrev_annotation_with_prompt_using_fulltext_df

,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,captured,captured as:,PMIDs from Pubtator3,Does alias represent gene in this publication?,How? I,How? II,"If not this gene, what is it?",Text Location,Relevant Text (surrounding),Notes,gene_name,pmid_full_text_chunk,context,alt_abbrev_prompt
0,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9477305,yes,Other,mouse phenotype biomarker,-,discussion,Mice homozygous for the tub (rd5) mutation exh...,NaN,TUB bipartite transcription factor,None,PMID: 9477305\n Full Text: None,Role:You are a biomedical gene nomenclature cu...
1,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9390831,yes,Other,mouse phenotype biomarker,-,abstract,Mice homozygous for a defect of the tub (rd5) ...,NaN,TUB bipartite transcription factor,These authors contributed equally to this work...,"PMID: 9390831\n Full Text: In this paper, we c...",Role:You are a biomedical gene nomenclature cu...
1,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9390831,yes,Other,mouse phenotype biomarker,-,abstract,Mice homozygous for a defect of the tub (rd5) ...,NaN,TUB bipartite transcription factor,"In this paper, we confirm that biallelic inact...","PMID: 9390831\n Full Text: In this paper, we c...",Role:You are a biomedical gene nomenclature cu...
2,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,7479945,yes,Other,mouse phenotype biomarker,-,abstract,We report a mouse model of type I Usher syndro...,NaN,TUB bipartite transcription factor,None,PMID: 7479945\n Full Text: None,Role:You are a biomedical gene nomenclature cu...
3,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,39961042,no,-,-,primer,methods,The primers used in this study were: RD5: 5′–A...,NaN,TUB bipartite transcription factor,Blastocystissp. is a zoonotic intestinal proto...,PMID: 39961042\n Full Text: Blastocystissp. is...,Role:You are a biomedical gene nomenclature cu...
4,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,38980911,no,-,-,primer,methods,PCR amplification of DNA was performed using a...,NaN,TUB bipartite transcription factor,The authors have declared that no competing in...,PMID: 38980911\n Full Text: • qPCR protocol\n\...,Role:You are a biomedical gene nomenclature cu...
4,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,38980911,no,-,-,primer,methods,PCR amplification of DNA was performed using a...,NaN,TUB bipartite transcription factor,"According to gender,BlastocystisST3, ST1, and ...",PMID: 38980911\n Full Text: • qPCR protocol\n\...,Role:You are a biomedical gene nomenclature cu...
4,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,38980911,no,-,-,primer,methods,PCR amplification of DNA was performed using a...,NaN,TUB bipartite transcription factor,"• qPCR protocol\n\n• nested-PCR protocol, incl...",PMID: 38980911\n Full Text: • qPCR protocol\n\...,Role:You are a biomedical gene nomenclature cu...
5,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,38578271,no,-,-,primer,methods,The gDNA samples were analysed by PCR amplific...,NaN,TUB bipartite transcription factor,Blastocystissp. is a zoonotic protozoan parasi...,PMID: 38578271\n Full Text: Blastocystissp. is...,Role:You are a biomedical gene nomenclature cu...
6,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,38543574,no,-,-,primer,methods,PCR amplification of Blastocystis sp. was carr...,NaN,TUB bipartite transcription factor,Blastocystissp. is the most common intestinal ...,PMID: 38543574\n Full Text: Blastocystissp. is...,Role:You are a biomedical gene nomenclature cu...


In [44]:
#Comment out after run!
test20_alt_abbrev_annotation_with_prompt_using_fulltext_df['alt_abbrev_response'] = None

for idx, row in test20_alt_abbrev_annotation_with_prompt_using_fulltext_df.iterrows():
    # Skip rows where there is no available full text
    if pd.isna(row['pmid_full_text_chunk']):
        continue

    test20_alt_abbrev_annotation_with_prompt_using_fulltext_df.at[
        idx, 'alt_abbrev_response'
    ] = query_claude_sonnet(row['alt_abbrev_prompt'])

    print(f'{idx} Done')

In [45]:
#Comment out after run!
test20_alt_abbrev_annotation_with_prompt_using_fulltext_df.to_excel('../output/test20_alt_abbrev_annotation_with_prompt_using_fulltext_df.xlsx',index=False)

In [46]:
test20_alt_abbrev_annotation_with_prompt_using_fulltext_df = pd.read_excel("../output/test20_alt_abbrev_annotation_with_prompt_using_fulltext_df.xlsx")

In [47]:
test20_alt_abbrev_annotation_with_prompt_using_fulltext_df

,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,captured,captured as:,PMIDs from Pubtator3,Does alias represent gene in this publication?,How?,"If not this gene, what is it?",Text Location,Relevant Text (surrounding),Notes,gene_name,pmid_full_text_chunk,context,alt_abbrev_prompt,alt_abbrev_response
0,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9477305,yes,mouse phenotype biomarker,-,discussion,Mice homozygous for the tub (rd5) mutation exh...,NaN,TUB bipartite transcription factor,NaN,PMID: 9477305\n Full Text: None,You are an expert biomedical scientist and gen...,NaN
1,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9390831,yes,mouse phenotype biomarker,-,abstract,Mice homozygous for a defect of the tub (rd5) ...,NaN,TUB bipartite transcription factor,These authors contributed equally to this work...,"PMID: 9390831\n Full Text: Conceptualization, ...",You are an expert biomedical scientist and gen...,"{\n ""pmid"": ""9390831"",\n ""alias_symbol"":..."
2,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9390831,yes,mouse phenotype biomarker,-,abstract,Mice homozygous for a defect of the tub (rd5) ...,NaN,TUB bipartite transcription factor,"Conceptualization, L.Z. and V.C.; methodology,...","PMID: 9390831\n Full Text: Conceptualization, ...",You are an expert biomedical scientist and gen...,"{\n ""pmid"": ""9390831"",\n ""alias_symbol"":..."
3,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,7479945,yes,mouse phenotype biomarker,-,abstract,We report a mouse model of type I Usher syndro...,NaN,TUB bipartite transcription factor,NaN,PMID: 7479945\n Full Text: None,You are an expert biomedical scientist and gen...,NaN
4,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,39961042,no,-,primer,methods,The primers used in this study were: RD5: 5′–A...,NaN,TUB bipartite transcription factor,Blastocystis sp. is a zoonotic intestinal prot...,PMID: 39961042\n Full Text: Blastocystis sp. i...,You are an expert biomedical scientist and gen...,"{\n ""pmid"": ""39961042"",\n ""alias_symbol""..."
5,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,38980911,no,-,primer,methods,PCR amplification of DNA was performed using a...,NaN,TUB bipartite transcription factor,The authors have declared that no competing in...,PMID: 38980911\n Full Text: When you are ready...,You are an expert biomedical scientist and gen...,"{\n ""pmid"": ""38980911"",\n ""alias_symbol"": ""r..."
6,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,38980911,no,-,primer,methods,PCR amplification of DNA was performed using a...,NaN,TUB bipartite transcription factor,"When you are ready to resubmit, please upload ...",PMID: 38980911\n Full Text: When you are ready...,You are an expert biomedical scientist and gen...,"{\n ""pmid"": ""38980911"",\n ""alias_symbol"": ""r..."
7,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,38578271,no,-,primer,methods,The gDNA samples were analysed by PCR amplific...,NaN,TUB bipartite transcription factor,Blastocystis sp. is a zoonotic protozoan paras...,PMID: 38578271\n Full Text: Blastocystis sp. i...,You are an expert biomedical scientist and gen...,"{\n ""pmid"": ""38578271"",\n ""alias_symbol""..."
8,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,38543574,no,-,primer,methods,PCR amplification of Blastocystis sp. was carr...,NaN,TUB bipartite transcription factor,Blastocystis sp. is the most common intestinal...,PMID: 38543574\n Full Text: Blastocystis sp. i...,You are an expert biomedical scientist and gen...,"{\n ""pmid"": ""38543574"",\n ""alias_symbol""..."
9,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,37185159,no,-,primer,methods,"In the first reaction, the broadly specific ol...",NaN,TUB bipartite transcription factor,Blastocystis is a protist of controversial pat...,PMID: 37185159\n Full Text: Fisher’s exact tes...,You are an expert biomedical scientist and gen...,"{\n ""pmid"": ""37185159"",\n ""alias_symbol""..."


#### <a id='toc1_1_5_2_'></a>[Run all samples (full text as context)](#toc0_)

In [48]:
# #Comment out after run!
# alt_abbrev_annotation_with_prompt_using_fulltext_df['alt_abbrev_response'] = None

# for idx, row in alt_abbrev_annotation_with_prompt_using_fulltext_df.iterrows():
#     # Skip rows where there is no available full text
#     if pd.isna(row['pmid_full_text_chunk']):
#         continue

#     alt_abbrev_annotation_with_prompt_using_fulltext_df.at[
#         idx, 'alt_abbrev_response'
#     ] = query_claude_sonnet(row['alt_abbrev_prompt'])

#     print(f'{idx} Done')

1 Done
1 Done
3 Done
4 Done
4 Done
4 Done
5 Done
6 Done
7 Done
7 Done
8 Done
8 Done
9 Done
11 Done
12 Done
12 Done
12 Done
14 Done
14 Done
14 Done
18 Done
18 Done
18 Done
19 Done
19 Done
20 Done
20 Done
20 Done
21 Done
21 Done
22 Done
22 Done
22 Done
24 Done
24 Done
24 Done
25 Done
25 Done
25 Done
25 Done
26 Done
26 Done
26 Done
26 Done
26 Done
26 Done
26 Done
27 Done
27 Done
27 Done
27 Done
28 Done
28 Done
28 Done
28 Done
28 Done
29 Done
29 Done
29 Done
29 Done
29 Done
30 Done
30 Done
30 Done
30 Done
30 Done
30 Done
31 Done
31 Done
31 Done
31 Done
31 Done
32 Done
32 Done
32 Done
33 Done
33 Done
33 Done
34 Done
34 Done
34 Done
34 Done
35 Done
35 Done
36 Done
36 Done
36 Done
36 Done
37 Done
38 Done
38 Done
38 Done
38 Done
38 Done
38 Done
38 Done
38 Done
38 Done
39 Done
39 Done
39 Done
41 Done
41 Done
41 Done
41 Done
41 Done
41 Done
42 Done
42 Done
42 Done
43 Done
43 Done
43 Done
43 Done
44 Done
44 Done
44 Done
45 Done
45 Done
45 Done
45 Done
45 Done
46 Done
46 Done
46 Done
47 Done
47 Do

In [49]:
# #Comment out after run!
# alt_abbrev_annotation_with_prompt_using_fulltext_df.to_excel('../output/alt_abbrev_annotation_with_prompt_using_fulltext_df.xlsx')

In [50]:
alt_abbrev_annotation_with_prompt_using_fulltext_df = pd.read_excel("../output/alt_abbrev_annotation_with_prompt_using_fulltext_df.xlsx")

In [52]:
alt_abbrev_annotation_with_prompt_using_fulltext_df

,Unnamed: 0,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,captured,captured as:,PMIDs from Pubtator3,Does alias represent gene in this publication?,...,How? II,"If not this gene, what is it?",Text Location,Relevant Text (surrounding),Notes,gene_name,pmid_full_text_chunk,context,alt_abbrev_prompt,alt_abbrev_response
0,0,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9477305,yes,...,mouse phenotype biomarker,-,discussion,Mice homozygous for the tub (rd5) mutation exh...,NaN,TUB bipartite transcription factor,NaN,PMID: 9477305\n Full Text: None,Role:You are a biomedical gene nomenclature cu...,NaN
1,1,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9390831,yes,...,mouse phenotype biomarker,-,abstract,Mice homozygous for a defect of the tub (rd5) ...,NaN,TUB bipartite transcription factor,These authors contributed equally to this work...,"PMID: 9390831\n Full Text: In this paper, we c...",Role:You are a biomedical gene nomenclature cu...,Here is the JSON output based on the provided ...
2,1,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9390831,yes,...,mouse phenotype biomarker,-,abstract,Mice homozygous for a defect of the tub (rd5) ...,NaN,TUB bipartite transcription factor,"In this paper, we confirm that biallelic inact...","PMID: 9390831\n Full Text: In this paper, we c...",Role:You are a biomedical gene nomenclature cu...,Here is the JSON output based on the provided ...
3,2,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,7479945,yes,...,mouse phenotype biomarker,-,abstract,We report a mouse model of type I Usher syndro...,NaN,TUB bipartite transcription factor,NaN,PMID: 7479945\n Full Text: None,Role:You are a biomedical gene nomenclature cu...,NaN
4,3,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,39961042,no,...,-,primer,methods,The primers used in this study were: RD5: 5′–A...,NaN,TUB bipartite transcription factor,Blastocystissp. is a zoonotic intestinal proto...,PMID: 39961042\n Full Text: Blastocystissp. is...,Role:You are a biomedical gene nomenclature cu...,"{\n""pmid"": ""39961042"",\n""alias_symbol"": ""rd5"",..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
360,124,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,26761027,no,...,-,genotype of rhinovirus,introduction,"A1A, A1B, A2, A23, A25, A29, A30, A31, A44, A4...",NaN,alpha-1-B glycoprotein,Rhinoviruses (RVs) and respiratory enterovirus...,PMID: 26761027\n Full Text: RVs are known to o...,Role:You are a biomedical gene nomenclature cu...,"{\n""pmid"": ""26761027"",\n""alias_symbol"": ""A1B"",..."
361,124,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,26761027,no,...,-,genotype of rhinovirus,introduction,"A1A, A1B, A2, A23, A25, A29, A30, A31, A44, A4...",NaN,alpha-1-B glycoprotein,Phylogeny of the different human enterovirus s...,PMID: 26761027\n Full Text: RVs are known to o...,Role:You are a biomedical gene nomenclature cu...,"{\n""pmid"": ""26761027"",\n""alias_symbol"": ""A1B"",..."
362,124,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,26761027,no,...,-,genotype of rhinovirus,introduction,"A1A, A1B, A2, A23, A25, A29, A30, A31, A44, A4...",NaN,alpha-1-B glycoprotein,RVs are known to optimally grow at cooler temp...,PMID: 26761027\n Full Text: RVs are known to o...,Role:You are a biomedical gene nomenclature cu...,"{\n""pmid"": ""26761027"",\n""alias_symbol"": ""A1B"",..."
363,125,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,31837028,no,...,-,red blood cell type,methods,We used in vitro phagocytosis by monocytes and...,NaN,alpha-1-B glycoprotein,Immunoglobulin therapy including intravenous i...,PMID: 31837028\n Full Text: Immunoglobulin the...,Role:You are a biomedical gene nomenclature cu...,Here is the JSON output based on the given inf...


In [53]:
#output should be none since there is no full text accessible programmatically for this sample
print(alt_abbrev_annotation_with_prompt_using_fulltext_df['alt_abbrev_response'][364])

nan


### Analysis (full text as content)

In [54]:
def parse_response(x):
    if pd.isna(x) or not str(x).strip():
        return np.nan
    try:
        return json.loads(x)
    except json.JSONDecodeError:
        return np.nan

In [55]:
analysis_alt_abbrev_annotation_with_prompt_using_fulltext_df = alt_abbrev_annotation_with_prompt_using_fulltext_df.copy()
analysis_alt_abbrev_annotation_with_prompt_using_fulltext_df["alt_abbrev_response_parsed"] = analysis_alt_abbrev_annotation_with_prompt_using_fulltext_df["alt_abbrev_response"].apply(parse_response)

In [56]:
def extract_representation_status(d):
    if not isinstance(d, dict):
        # response was empty / invalid
        return np.nan

    if "alias_symbol_represents_primary_gene" not in d:
        return np.nan

    if d["alias_symbol_represents_primary_gene"] == "":
        # explicit empty string in JSON
        return "-"

    return d["alias_symbol_represents_primary_gene"]

In [57]:
analysis_alt_abbrev_annotation_with_prompt_using_fulltext_df["response_alias_symbol_represents_primary_gene"] = analysis_alt_abbrev_annotation_with_prompt_using_fulltext_df["alt_abbrev_response_parsed"].apply(
    extract_representation_status
)

In [58]:
def extract_relationship_type(d):
    if not isinstance(d, dict):
        # response was empty / invalid
        return np.nan

    if "relationship_type" not in d:
        return np.nan

    if d["relationship_type"] == "":
        # explicit empty string in JSON
        return "-"

    return d["relationship_type"]

In [59]:
analysis_alt_abbrev_annotation_with_prompt_using_fulltext_df["response_relationship_type"] = analysis_alt_abbrev_annotation_with_prompt_using_fulltext_df["alt_abbrev_response_parsed"].apply(
    extract_relationship_type
)

In [60]:
analysis_alt_abbrev_annotation_with_prompt_using_fulltext_df["response_evidence"] = (
    analysis_alt_abbrev_annotation_with_prompt_using_fulltext_df["alt_abbrev_response_parsed"]
    .apply(lambda d: d.get("evidence") if isinstance(d, dict) else None)
)

In [61]:
analysis_alt_abbrev_annotation_with_prompt_using_fulltext_df

,Unnamed: 0,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,captured,captured as:,PMIDs from Pubtator3,Does alias represent gene in this publication?,...,Notes,gene_name,pmid_full_text_chunk,context,alt_abbrev_prompt,alt_abbrev_response,alt_abbrev_response_parsed,response_alias_symbol_represents_primary_gene,response_relationship_type,response_evidence
0,0,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9477305,yes,...,NaN,TUB bipartite transcription factor,NaN,PMID: 9477305\n Full Text: None,Role:You are a biomedical gene nomenclature cu...,NaN,NaN,NaN,NaN,None
1,1,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9390831,yes,...,NaN,TUB bipartite transcription factor,These authors contributed equally to this work...,"PMID: 9390831\n Full Text: In this paper, we c...",Role:You are a biomedical gene nomenclature cu...,Here is the JSON output based on the provided ...,NaN,NaN,NaN,None
2,1,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9390831,yes,...,NaN,TUB bipartite transcription factor,"In this paper, we confirm that biallelic inact...","PMID: 9390831\n Full Text: In this paper, we c...",Role:You are a biomedical gene nomenclature cu...,Here is the JSON output based on the provided ...,NaN,NaN,NaN,None
3,2,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,7479945,yes,...,NaN,TUB bipartite transcription factor,NaN,PMID: 7479945\n Full Text: None,Role:You are a biomedical gene nomenclature cu...,NaN,NaN,NaN,NaN,None
4,3,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,39961042,no,...,NaN,TUB bipartite transcription factor,Blastocystissp. is a zoonotic intestinal proto...,PMID: 39961042\n Full Text: Blastocystissp. is...,Role:You are a biomedical gene nomenclature cu...,"{\n""pmid"": ""39961042"",\n""alias_symbol"": ""rd5"",...","{'pmid': '39961042', 'alias_symbol': 'rd5', 'p...",NO,-,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
360,124,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,26761027,no,...,NaN,alpha-1-B glycoprotein,Rhinoviruses (RVs) and respiratory enterovirus...,PMID: 26761027\n Full Text: RVs are known to o...,Role:You are a biomedical gene nomenclature cu...,"{\n""pmid"": ""26761027"",\n""alias_symbol"": ""A1B"",...","{'pmid': '26761027', 'alias_symbol': 'A1B', 'p...",NO,-,
361,124,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,26761027,no,...,NaN,alpha-1-B glycoprotein,Phylogeny of the different human enterovirus s...,PMID: 26761027\n Full Text: RVs are known to o...,Role:You are a biomedical gene nomenclature cu...,"{\n""pmid"": ""26761027"",\n""alias_symbol"": ""A1B"",...","{'pmid': '26761027', 'alias_symbol': 'A1B', 'p...",NO,-,
362,124,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,26761027,no,...,NaN,alpha-1-B glycoprotein,RVs are known to optimally grow at cooler temp...,PMID: 26761027\n Full Text: RVs are known to o...,Role:You are a biomedical gene nomenclature cu...,"{\n""pmid"": ""26761027"",\n""alias_symbol"": ""A1B"",...","{'pmid': '26761027', 'alias_symbol': 'A1B', 'p...",NO,-,
363,125,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,31837028,no,...,NaN,alpha-1-B glycoprotein,Immunoglobulin therapy including intravenous i...,PMID: 31837028\n Full Text: Immunoglobulin the...,Role:You are a biomedical gene nomenclature cu...,Here is the JSON output based on the given inf...,NaN,NaN,NaN,None


In [62]:
analysis_alt_abbrev_annotation_with_prompt_using_fulltext_df["represent_primary_gene_matches"] = (
    analysis_alt_abbrev_annotation_with_prompt_using_fulltext_df["Does alias represent gene in this publication?"].str.strip().str.upper()
    == analysis_alt_abbrev_annotation_with_prompt_using_fulltext_df["response_alias_symbol_represents_primary_gene"].str.strip().str.upper()
)

analysis_alt_abbrev_annotation_with_prompt_using_fulltext_df.loc[analysis_alt_abbrev_annotation_with_prompt_using_fulltext_df["alt_abbrev_response"].isna(), "represent_primary_gene_matches"] = np.nan


/var/folders/vt/znzp_c6s02q6kjcmqfk0cb500000gq/T/ipykernel_99933/2032004202.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'nan' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  analysis_alt_abbrev_annotation_with_prompt_using_fulltext_df.loc[analysis_alt_abbrev_annotation_with_prompt_using_fulltext_df["alt_abbrev_response"].isna(), "represent_primary_gene_matches"] = np.nan


In [63]:
analysis_alt_abbrev_annotation_with_prompt_using_fulltext_df["relationship_type_matches"] = (
    analysis_alt_abbrev_annotation_with_prompt_using_fulltext_df["How? I"].str.strip().str.upper()
    == analysis_alt_abbrev_annotation_with_prompt_using_fulltext_df["response_relationship_type"].str.strip().str.upper()
)

analysis_alt_abbrev_annotation_with_prompt_using_fulltext_df.loc[analysis_alt_abbrev_annotation_with_prompt_using_fulltext_df["alt_abbrev_response"].isna(), "relationship_type_matches"] = np.nan


/var/folders/vt/znzp_c6s02q6kjcmqfk0cb500000gq/T/ipykernel_99933/2922950614.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'nan' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  analysis_alt_abbrev_annotation_with_prompt_using_fulltext_df.loc[analysis_alt_abbrev_annotation_with_prompt_using_fulltext_df["alt_abbrev_response"].isna(), "relationship_type_matches"] = np.nan


In [64]:
analysis_alt_abbrev_annotation_with_prompt_using_fulltext_df

,Unnamed: 0,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,captured,captured as:,PMIDs from Pubtator3,Does alias represent gene in this publication?,...,pmid_full_text_chunk,context,alt_abbrev_prompt,alt_abbrev_response,alt_abbrev_response_parsed,response_alias_symbol_represents_primary_gene,response_relationship_type,response_evidence,represent_primary_gene_matches,relationship_type_matches
0,0,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9477305,yes,...,NaN,PMID: 9477305\n Full Text: None,Role:You are a biomedical gene nomenclature cu...,NaN,NaN,NaN,NaN,None,NaN,NaN
1,1,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9390831,yes,...,These authors contributed equally to this work...,"PMID: 9390831\n Full Text: In this paper, we c...",Role:You are a biomedical gene nomenclature cu...,Here is the JSON output based on the provided ...,NaN,NaN,NaN,None,False,False
2,1,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9390831,yes,...,"In this paper, we confirm that biallelic inact...","PMID: 9390831\n Full Text: In this paper, we c...",Role:You are a biomedical gene nomenclature cu...,Here is the JSON output based on the provided ...,NaN,NaN,NaN,None,False,False
3,2,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,7479945,yes,...,NaN,PMID: 7479945\n Full Text: None,Role:You are a biomedical gene nomenclature cu...,NaN,NaN,NaN,NaN,None,NaN,NaN
4,3,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,39961042,no,...,Blastocystissp. is a zoonotic intestinal proto...,PMID: 39961042\n Full Text: Blastocystissp. is...,Role:You are a biomedical gene nomenclature cu...,"{\n""pmid"": ""39961042"",\n""alias_symbol"": ""rd5"",...","{'pmid': '39961042', 'alias_symbol': 'rd5', 'p...",NO,-,,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
360,124,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,26761027,no,...,Rhinoviruses (RVs) and respiratory enterovirus...,PMID: 26761027\n Full Text: RVs are known to o...,Role:You are a biomedical gene nomenclature cu...,"{\n""pmid"": ""26761027"",\n""alias_symbol"": ""A1B"",...","{'pmid': '26761027', 'alias_symbol': 'A1B', 'p...",NO,-,,True,True
361,124,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,26761027,no,...,Phylogeny of the different human enterovirus s...,PMID: 26761027\n Full Text: RVs are known to o...,Role:You are a biomedical gene nomenclature cu...,"{\n""pmid"": ""26761027"",\n""alias_symbol"": ""A1B"",...","{'pmid': '26761027', 'alias_symbol': 'A1B', 'p...",NO,-,,True,True
362,124,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,26761027,no,...,RVs are known to optimally grow at cooler temp...,PMID: 26761027\n Full Text: RVs are known to o...,Role:You are a biomedical gene nomenclature cu...,"{\n""pmid"": ""26761027"",\n""alias_symbol"": ""A1B"",...","{'pmid': '26761027', 'alias_symbol': 'A1B', 'p...",NO,-,,True,True
363,125,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,31837028,no,...,Immunoglobulin therapy including intravenous i...,PMID: 31837028\n Full Text: Immunoglobulin the...,Role:You are a biomedical gene nomenclature cu...,Here is the JSON output based on the given inf...,NaN,NaN,NaN,None,False,False


In [65]:
def consolidate_match(s):
    if s.eq(True).any():
        return True
    if s.eq(False).any():
        return False
    return np.nan


In [66]:
cols = [
    "represent_primary_gene_matches",
    "relationship_type_matches",
]

consolidated_analysis_alt_abbrev_annotation_with_prompt_using_fulltext_df = (
    analysis_alt_abbrev_annotation_with_prompt_using_fulltext_df
    .groupby("PMIDs from Pubtator3", as_index=False)[cols]
    .agg(consolidate_match)
)

In [67]:
consolidated_analysis_alt_abbrev_annotation_with_prompt_using_fulltext_df

,PMIDs from Pubtator3,represent_primary_gene_matches,relationship_type_matches
0,2447112,NaN,NaN
1,6322416,True,True
2,7479945,NaN,NaN
3,7486253,NaN,NaN
4,7506283,True,True
...,...,...,...
122,40662138,False,False
123,40721578,False,False
124,40819127,False,False
125,40890122,False,False


In [68]:
def create_analysis_summary(column: pd.Series):
    """ Summarize value counts with:
      - True/False percentages over non-NaN denominator
      - NaN percentage over total rows

    :param column: Dataframe and column to analyze (Example: df["column"])
    :return: dataframe with summarized results
    """
    counts = (
        column.value_counts(dropna=False)
              .rename("numerator")
              .rename_axis("consensus_w_curator")
    )

    non_nan_denominator = column.notna().sum()
    total_denominator = len(column)

    return (
        counts
        .reset_index()
        .assign(
            denominator=lambda df: df["consensus_w_curator"].apply(
                lambda x: total_denominator if pd.isna(x) else non_nan_denominator
            ),
            percentage=lambda df: df["numerator"] / df["denominator"] * 100
        )
    )

In [69]:
represent_primary_gene_match_analysis_summary = create_analysis_summary(
    consolidated_analysis_alt_abbrev_annotation_with_prompt_using_fulltext_df["represent_primary_gene_matches"]
)
represent_primary_gene_match_analysis_summary

,consensus_w_curator,numerator,denominator,percentage
0,False,63,113,55.752212
1,True,50,113,44.247788
2,NaN,14,127,11.023622


In [70]:
relationship_type_match_analysis_summary = create_analysis_summary(
    consolidated_analysis_alt_abbrev_annotation_with_prompt_using_fulltext_df["relationship_type_matches"]
)
relationship_type_match_analysis_summary

,consensus_w_curator,numerator,denominator,percentage
0,False,68,113,60.176991
1,True,45,113,39.823009
2,NaN,14,127,11.023622


### Analysis (20 samples, full text as content)

In [ ]:
analysis_test20_alt_abbrev_annotation_with_prompt_using_fulltext_df = test20_alt_abbrev_annotation_with_prompt_using_fulltext_df.copy()
analysis_test20_alt_abbrev_annotation_with_prompt_using_fulltext_df["alt_abbrev_response_parsed"] = analysis_test20_alt_abbrev_annotation_with_prompt_using_fulltext_df["alt_abbrev_response"].apply(parse_response)

In [ ]:
def extract_representation_status(d):
    if not isinstance(d, dict):
        # response was empty / invalid
        return np.nan

    if "alias_symbol_represents_primary_gene" not in d:
        return np.nan

    if d["alias_symbol_represents_primary_gene"] == "":
        # explicit empty string in JSON
        return "-"

    return d["alias_symbol_represents_primary_gene"]

In [ ]:
analysis_test20_alt_abbrev_annotation_with_prompt_using_fulltext_df["response_alias_symbol_represents_primary_gene"] = analysis_test20_alt_abbrev_annotation_with_prompt_using_fulltext_df["alt_abbrev_response_parsed"].apply(
    extract_representation_status
)

In [ ]:
analysis_test20_alt_abbrev_annotation_with_prompt_using_fulltext_df["response_relationship_type"] = analysis_test20_alt_abbrev_annotation_with_prompt_using_fulltext_df["alt_abbrev_response_parsed"].apply(
    extract_relationship_type
)

In [ ]:
analysis_test20_alt_abbrev_annotation_with_prompt_using_fulltext_df["response_evidence"] = (
    analysis_test20_alt_abbrev_annotation_with_prompt_using_fulltext_df["alt_abbrev_response_parsed"]
    .apply(lambda d: d.get("evidence") if isinstance(d, dict) else None)
)

In [ ]:
analysis_test20_alt_abbrev_annotation_with_prompt_using_fulltext_df

,Unnamed: 0,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,captured,captured as:,PMIDs from Pubtator3,Does alias represent gene in this publication?,...,Notes,gene_name,pmid_full_text_chunk,context,alt_abbrev_prompt,alt_abbrev_response,alt_abbrev_response_parsed,response_alias_symbol_represents_primary_gene,response_relationship_type,response_evidence
0,0,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9477305,yes,...,NaN,TUB bipartite transcription factor,NaN,PMID: 9477305\n Full Text: None,Role:You are a biomedical gene nomenclature cu...,NaN,NaN,NaN,NaN,None
1,1,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9390831,yes,...,NaN,TUB bipartite transcription factor,These authors contributed equally to this work...,"PMID: 9390831\n Full Text: In this paper, we c...",Role:You are a biomedical gene nomenclature cu...,Here is the JSON output based on the provided ...,NaN,NaN,NaN,None
2,1,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9390831,yes,...,NaN,TUB bipartite transcription factor,"In this paper, we confirm that biallelic inact...","PMID: 9390831\n Full Text: In this paper, we c...",Role:You are a biomedical gene nomenclature cu...,Here is the JSON output based on the provided ...,NaN,NaN,NaN,None
3,2,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,7479945,yes,...,NaN,TUB bipartite transcription factor,NaN,PMID: 7479945\n Full Text: None,Role:You are a biomedical gene nomenclature cu...,NaN,NaN,NaN,NaN,None
4,3,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,39961042,no,...,NaN,TUB bipartite transcription factor,Blastocystissp. is a zoonotic intestinal proto...,PMID: 39961042\n Full Text: Blastocystissp. is...,Role:You are a biomedical gene nomenclature cu...,"{\n""pmid"": ""39961042"",\n""alias_symbol"": ""rd5"",...","{'pmid': '39961042', 'alias_symbol': 'rd5', 'p...",NO,-,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
360,124,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,26761027,no,...,NaN,alpha-1-B glycoprotein,Rhinoviruses (RVs) and respiratory enterovirus...,PMID: 26761027\n Full Text: RVs are known to o...,Role:You are a biomedical gene nomenclature cu...,"{\n""pmid"": ""26761027"",\n""alias_symbol"": ""A1B"",...","{'pmid': '26761027', 'alias_symbol': 'A1B', 'p...",NO,-,
361,124,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,26761027,no,...,NaN,alpha-1-B glycoprotein,Phylogeny of the different human enterovirus s...,PMID: 26761027\n Full Text: RVs are known to o...,Role:You are a biomedical gene nomenclature cu...,"{\n""pmid"": ""26761027"",\n""alias_symbol"": ""A1B"",...","{'pmid': '26761027', 'alias_symbol': 'A1B', 'p...",NO,-,
362,124,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,26761027,no,...,NaN,alpha-1-B glycoprotein,RVs are known to optimally grow at cooler temp...,PMID: 26761027\n Full Text: RVs are known to o...,Role:You are a biomedical gene nomenclature cu...,"{\n""pmid"": ""26761027"",\n""alias_symbol"": ""A1B"",...","{'pmid': '26761027', 'alias_symbol': 'A1B', 'p...",NO,-,
363,125,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,31837028,no,...,NaN,alpha-1-B glycoprotein,Immunoglobulin therapy including intravenous i...,PMID: 31837028\n Full Text: Immunoglobulin the...,Role:You are a biomedical gene nomenclature cu...,Here is the JSON output based on the given inf...,NaN,NaN,NaN,None


In [ ]:
analysis_test20_alt_abbrev_annotation_with_prompt_using_fulltext_df["represent_primary_gene_matches"] = (
    analysis_test20_alt_abbrev_annotation_with_prompt_using_fulltext_df["Does alias represent gene in this publication?"].str.strip().str.upper()
    == analysis_test20_alt_abbrev_annotation_with_prompt_using_fulltext_df["response_alias_symbol_represents_primary_gene"].str.strip().str.upper()
)

analysis_test20_alt_abbrev_annotation_with_prompt_using_fulltext_df.loc[analysis_test20_alt_abbrev_annotation_with_prompt_using_fulltext_df["alt_abbrev_response"].isna(), "represent_primary_gene_matches"] = np.nan


/var/folders/vt/znzp_c6s02q6kjcmqfk0cb500000gq/T/ipykernel_99933/2032004202.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'nan' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  analysis_alt_abbrev_annotation_with_prompt_using_fulltext_df.loc[analysis_alt_abbrev_annotation_with_prompt_using_fulltext_df["alt_abbrev_response"].isna(), "represent_primary_gene_matches"] = np.nan


In [ ]:
analysis_test20_alt_abbrev_annotation_with_prompt_using_fulltext_df["relationship_type_matches"] = (
    analysis_test20_alt_abbrev_annotation_with_prompt_using_fulltext_df["How? I"].str.strip().str.upper()
    == analysis_test20_alt_abbrev_annotation_with_prompt_using_fulltext_df["response_relationship_type"].str.strip().str.upper()
)

analysis_test20_alt_abbrev_annotation_with_prompt_using_fulltext_df.loc[analysis_test20_alt_abbrev_annotation_with_prompt_using_fulltext_df["alt_abbrev_response"].isna(), "relationship_type_matches"] = np.nan


/var/folders/vt/znzp_c6s02q6kjcmqfk0cb500000gq/T/ipykernel_99933/2922950614.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'nan' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  analysis_alt_abbrev_annotation_with_prompt_using_fulltext_df.loc[analysis_alt_abbrev_annotation_with_prompt_using_fulltext_df["alt_abbrev_response"].isna(), "relationship_type_matches"] = np.nan


In [ ]:
analysis_test20_alt_abbrev_annotation_with_prompt_using_fulltext_df

,Unnamed: 0,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,captured,captured as:,PMIDs from Pubtator3,Does alias represent gene in this publication?,...,pmid_full_text_chunk,context,alt_abbrev_prompt,alt_abbrev_response,alt_abbrev_response_parsed,response_alias_symbol_represents_primary_gene,response_relationship_type,response_evidence,represent_primary_gene_matches,relationship_type_matches
0,0,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9477305,yes,...,NaN,PMID: 9477305\n Full Text: None,Role:You are a biomedical gene nomenclature cu...,NaN,NaN,NaN,NaN,None,NaN,NaN
1,1,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9390831,yes,...,These authors contributed equally to this work...,"PMID: 9390831\n Full Text: In this paper, we c...",Role:You are a biomedical gene nomenclature cu...,Here is the JSON output based on the provided ...,NaN,NaN,NaN,None,False,False
2,1,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9390831,yes,...,"In this paper, we confirm that biallelic inact...","PMID: 9390831\n Full Text: In this paper, we c...",Role:You are a biomedical gene nomenclature cu...,Here is the JSON output based on the provided ...,NaN,NaN,NaN,None,False,False
3,2,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,7479945,yes,...,NaN,PMID: 7479945\n Full Text: None,Role:You are a biomedical gene nomenclature cu...,NaN,NaN,NaN,NaN,None,NaN,NaN
4,3,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,39961042,no,...,Blastocystissp. is a zoonotic intestinal proto...,PMID: 39961042\n Full Text: Blastocystissp. is...,Role:You are a biomedical gene nomenclature cu...,"{\n""pmid"": ""39961042"",\n""alias_symbol"": ""rd5"",...","{'pmid': '39961042', 'alias_symbol': 'rd5', 'p...",NO,-,,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
360,124,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,26761027,no,...,Rhinoviruses (RVs) and respiratory enterovirus...,PMID: 26761027\n Full Text: RVs are known to o...,Role:You are a biomedical gene nomenclature cu...,"{\n""pmid"": ""26761027"",\n""alias_symbol"": ""A1B"",...","{'pmid': '26761027', 'alias_symbol': 'A1B', 'p...",NO,-,,True,True
361,124,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,26761027,no,...,Phylogeny of the different human enterovirus s...,PMID: 26761027\n Full Text: RVs are known to o...,Role:You are a biomedical gene nomenclature cu...,"{\n""pmid"": ""26761027"",\n""alias_symbol"": ""A1B"",...","{'pmid': '26761027', 'alias_symbol': 'A1B', 'p...",NO,-,,True,True
362,124,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,26761027,no,...,RVs are known to optimally grow at cooler temp...,PMID: 26761027\n Full Text: RVs are known to o...,Role:You are a biomedical gene nomenclature cu...,"{\n""pmid"": ""26761027"",\n""alias_symbol"": ""A1B"",...","{'pmid': '26761027', 'alias_symbol': 'A1B', 'p...",NO,-,,True,True
363,125,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,31837028,no,...,Immunoglobulin therapy including intravenous i...,PMID: 31837028\n Full Text: Immunoglobulin the...,Role:You are a biomedical gene nomenclature cu...,Here is the JSON output based on the given inf...,NaN,NaN,NaN,None,False,False


In [ ]:
cols = [
    "represent_primary_gene_matches",
    "relationship_type_matches",
]

consolidated_analysis_test20_alt_abbrev_annotation_with_prompt_using_fulltext_df = (
    analysis_test20_alt_abbrev_annotation_with_prompt_using_fulltext_df
    .groupby("PMIDs from Pubtator3", as_index=False)[cols]
    .agg(consolidate_match)
)

In [ ]:
consolidated_analysis_test20_alt_abbrev_annotation_with_prompt_using_fulltext_df

,PMIDs from Pubtator3,represent_primary_gene_matches,relationship_type_matches
0,2447112,NaN,NaN
1,6322416,True,True
2,7479945,NaN,NaN
3,7486253,NaN,NaN
4,7506283,True,True
...,...,...,...
122,40662138,False,False
123,40721578,False,False
124,40819127,False,False
125,40890122,False,False


In [ ]:
represent_primary_gene_match_analysis_summary = create_analysis_summary(
    consolidated_analysis_test20_alt_abbrev_annotation_with_prompt_using_fulltext_df["represent_primary_gene_matches"]
)
represent_primary_gene_match_analysis_summary

,consensus_w_curator,numerator,denominator,percentage
0,False,63,113,55.752212
1,True,50,113,44.247788
2,NaN,14,127,11.023622


In [ ]:
relationship_type_match_analysis_summary = create_analysis_summary(
    consolidated_analysis_test20_alt_abbrev_annotation_with_prompt_using_fulltext_df["relationship_type_matches"]
)
relationship_type_match_analysis_summary

,consensus_w_curator,numerator,denominator,percentage
0,False,68,113,60.176991
1,True,45,113,39.823009
2,NaN,14,127,11.023622
